# Analyse des alertes CSIRT
Quatre flux RSS, collecte bridée, regroupements et statuts éditoriaux. Les cellules consultent la base locale sans lancer de collecte. Les essais réseau sont explicites et utilisent les limites du collecteur.


In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
from csirt.config import Settings, load_sources
settings=Settings.from_env()
assert settings.db_path.is_file(), "Lancer le collecteur au préalable"
db=sqlite3.connect(settings.db_path.resolve().as_uri()+"?mode=ro",uri=True)
pd.DataFrame([{k:s[k] for k in ("id","name","poll_seconds","enabled")} for s in load_sources(settings)])

In [ ]:
pd.read_sql_query("SELECT s.name,COUNT(o.id) publications,MAX(o.published_at) derniere_publication,s.last_success_at,s.failures FROM sources s LEFT JOIN observations o ON o.source_id=s.id GROUP BY s.id",db)

In [ ]:
incidents=pd.read_sql_query("SELECT i.id,i.organization,i.reference_at,i.first_seen_at,i.last_changed_at,COUNT(o.id) publications,COUNT(DISTINCT o.source_id) sources FROM incidents i JOIN observations o ON o.incident_id=i.id GROUP BY i.id ORDER BY i.reference_at DESC",db)
incidents.head(30)

In [ ]:
statuts=pd.read_sql_query("SELECT s.name source,o.status,COUNT(*) nombre FROM observations o JOIN sources s ON s.id=o.source_id GROUP BY s.id,o.status",db)
statuts.pivot(index="source",columns="status",values="nombre").fillna(0)

In [ ]:
incidents[incidents.sources>1].head(30)

In [ ]:
publications=pd.read_sql_query("SELECT published_at,source_id,status FROM observations",db)
publications["date"]=pd.to_datetime(publications.published_at,utc=True).dt.date
publications.groupby(["date","source_id"]).size().unstack(fill_value=0).tail(30)

## Contrôler le parseur sans solliciter les sources
Utiliser un flux enregistré ou un exemple local ; aucune requête réseau dans cette cellule.


In [ ]:
from csirt.feeds import parse_feed
exemple=b'<rss><channel><item><title>Exemple</title><guid>demo</guid><link>https://example.org/alerte</link></item></channel></rss>'
parse_feed(exemple,"local-test",settings.max_entries).entries

In [ ]:
db.close()